# Capítulo 2 — O Primeiro Cálculo com PySCF

**Simulando Qubits com PySCF** · Cristiano Alves

Neste *notebook* executamos, do zero, cálculos Hartree-Fock reais: o átomo de hidrogênio, a molécula H₂ e — como primeiro passo concreto do Projeto Âncora — o átomo de silício. Execute as células **na ordem** (`Shift+Enter`).

> 💡 Se o PySCF ainda não estiver instalado neste ambiente, descomente e execute a célula abaixo.

In [1]:
# !pip install pyscf

## 2.1 A anatomia de um cálculo em PySCF

Quase todo cálculo segue três passos:

1. **Definir o sistema** — objeto `Mole` (`gto.M`).
2. **Escolher o método** — objeto `SCF` (aqui, Hartree-Fock).
3. **Executar e analisar** — `kernel()` e leitura dos resultados.

Em código, o esqueleto cabe em cinco linhas:

In [2]:
from pyscf import gto, scf          # 1. importa os modulos

mol = gto.M(atom='H 0 0 0; H 0 0 0.74',  # 2. define o sistema
            basis='sto-3g')

mf = scf.RHF(mol)                   # 3. escolhe o metodo (Hartree-Fock)
energia = mf.kernel()               # 4. executa e obtem a energia total

converged SCF energy = -1.11675930739643


## 2.2 Definindo uma molécula: o objeto `Mole`

A função `gto.M()` reúne geometria, base, carga e spin. A geometria vai no argumento `atom` (símbolo + x y z, em ångström por padrão).

> ⚠️ **Atenção:** `spin` **não** é a multiplicidade; é o **número de elétrons desemparelhados** ($n_\alpha - n_\beta = 2S$). Camada fechada → `spin=0`; átomo de H → `spin=1`; átomo de Si → `spin=2`.

In [3]:
from pyscf import gto

mol = gto.M(
    atom="""H  0.0  0.0  0.0
            H  0.0  0.0  0.74""",   # geometria: simbolo x y z (por linha)
    basis='sto-3g',                 # conjunto de funcoes de base
    charge=0,                       # carga total do sistema
    spin=0,                         # numero de eletrons desemparelhados (2S)
    unit='Angstrom',                # unidade das coordenadas (padrao)
    verbose=0,                      # nivel de detalhe da saida (0 a 9)
)

### Interrogando o objeto `Mole`

Vale inspecionar o sistema antes de calcular — é uma checagem barata que evita erros.

In [4]:
mol = gto.M(atom='H 0 0 0; H 0 0 0.74', basis='sto-3g')

print("Numero de eletrons:", mol.nelectron)         # 2
print("Numero de atomos:", mol.natm)                # 2
print("Numero de funcoes de base (AOs):", mol.nao)  # 2
print("Carga:", mol.charge, " Spin (2S):", mol.spin)  # 0  0

Numero de eletrons: 2
Numero de atomos: 2
Numero de funcoes de base (AOs): 2
Carga: 0  Spin (2S): 0


## 2.3 O que é uma base?

Cada orbital molecular é escrito como combinação linear de funções fixas — as **funções de base**:
$$\psi_i(\mathbf{r}) = \sum_{\mu} C_{\mu i}\,\phi_\mu(\mathbf{r}).$$
Bases maiores → mais precisas → mais caras. Famílias comuns: **STO-3G** (mínima), **cc-pVDZ/cc-pVTZ** (correlação consistente), **def2-SVP** (Karlsruhe).

### O efeito da base na prática

Observe a energia HF da H₂ **descer** (aproximando-se do limite ≈ −1,1336 Ha) à medida que a base cresce.

In [5]:
from pyscf import gto, scf

print("%-10s %-8s %s" % ("Base", "nao", "Energia (Ha)"))
for base in ['sto-3g', 'cc-pvdz', 'cc-pvtz']:
    mol = gto.M(atom='H 0 0 0; H 0 0 0.74', basis=base, verbose=0)
    e = scf.RHF(mol).kernel()
    print("%-10s %-8d %.6f" % (base, mol.nao, e))

Base       nao      Energia (Ha)
sto-3g     2        -1.116759
cc-pvdz    10       -1.128700
cc-pvtz    28       -1.132968


> 💡 **Princípio variacional:** a energia HF é sempre um limite **superior** à energia verdadeira. Logo, entre duas bases, a que dá energia **mais baixa** é a mais próxima da realidade.

## 2.4 Executando Hartree-Fock: o objeto `SCF`

O **Hartree-Fock** aproxima a repulsão elétron-elétron por um campo médio, resolvido de forma iterativa e autoconsistente (*Self-Consistent Field*). Três sabores:

- `scf.RHF` — *Restricted*: camada fechada (`spin=0`), como a H₂.
- `scf.ROHF` — *Restricted Open-shell*: átomos e radicais (H, Si).
- `scf.UHF` — *Unrestricted*: radicais e acoplamento de troca (Parte III).

`kernel()` dispara o cálculo e devolve a energia total (também em `mf.e_tot`).

In [6]:
from pyscf import gto, scf

mol = gto.M(atom='H 0 0 0; H 0 0 0.74', basis='sto-3g')

mf = scf.RHF(mol)
energia = mf.kernel()

print("Energia total: %.8f Ha" % energia)
print("Confirmando via atributo: %.8f Ha" % mf.e_tot)

converged SCF energy = -1.11675930739643


Energia total: -1.11675931 Ha
Confirmando via atributo: -1.11675931 Ha


## 2.5 Interpretando a saída

Com `verbose=4`, o PySCF revela o andamento do ciclo SCF. Usemos a molécula de água, que leva alguns ciclos para convergir (a H₂, minúscula, converge quase de imediato).

Observe as colunas: `E` (energia do ciclo, descendo), `delta_E` (variação — critério principal, ~$10^{-9}$ Ha na convergência) e `|g|` (norma do gradiente).

In [7]:
from pyscf import gto, scf

mol = gto.M(atom="""O  0.000  0.000  0.117
                    H  0.000  0.757 -0.469
                    H  0.000 -0.757 -0.469""",
            basis='sto-3g', verbose=4)

mf = scf.RHF(mol)
mf.kernel()

System: uname_result(system='Linux', node='cristianoalves-nitro-pro-n7', release='6.8.0-134-generic', version='#134-Ubuntu SMP PREEMPT_DYNAMIC Fri Jun 26 18:43:11 UTC 2026', machine='x86_64')  Threads 16
Python 3.11.7 (main, Dec 15 2023, 18:12:31) [GCC 11.2.0]
numpy 2.4.4  scipy 1.16.2  h5py 3.14.0
Date: Tue Jul 21 10:30:05 2026
PySCF version 2.14.0
PySCF path  /home/cristianoalves/anaconda3/lib/python3.11/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 3
[INPUT] num. electrons = 10
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 O      0.000000000000   0.000000000000   0.117000000000 AA    0.000000000000   0.000000000000   0.221097956574 Bohr   0.0
[INPUT]  2 H      0.000000000000   0.757000000000  -0.469000000000 AA    0.000

number of shells = 5


number of NR pGTOs = 21


number of NR cGTOs = 7


basis = sto-3g


...

cycle= 6 E= -74.9629466565312  delta_E= -1.58e-10  |g|= 3.45e-06  |ddm|= 4.24e-05


  HOMO = -0.391242366126425  LUMO = 0.605576214435601  gap/eV = 27.12482


Extra cycle  E= -74.9629466565387  delta_E= -7.47e-12  |g|= 1.47e-06  |ddm|= 7.42e-06


converged SCF energy = -74.9629466565387


np.float64(-74.9629466565387)

> ⚠️ **Convergiu ≠ correto.** `converged SCF energy` garante apenas que o algoritmo achou uma solução autoconsistente. E se aparecer `SCF not converged`, o número **não tem valor**. Sempre confira a convergência.

### Energias orbitais, ocupações e o *gap*

Os atributos `mf.mo_energy` (energias, em ordem crescente) e `mf.mo_occ` (ocupações) descrevem os orbitais moleculares.

In [8]:
import numpy as np

print("Energias orbitais (Ha):", np.round(mf.mo_energy, 4))
print("Ocupacoes:", mf.mo_occ)

# HOMO = ultimo orbital ocupado; LUMO = primeiro vazio
n_ocupados = int(mf.mo_occ.sum() // 2)      # 5 orbitais duplamente ocupados
homo = mf.mo_energy[n_ocupados - 1]
lumo = mf.mo_energy[n_ocupados]
gap = lumo - homo

print("HOMO: %.4f Ha   LUMO: %.4f Ha" % (homo, lumo))
print("Gap HOMO-LUMO: %.4f Ha = %.2f eV" % (gap, gap * 27.2114))

Energias orbitais (Ha): [-20.2418  -1.2684  -0.6179  -0.453   -0.3912   0.6056   0.7422]
Ocupacoes: [2. 2. 2. 2. 2. 0. 0.]
HOMO: -0.3912 Ha   LUMO: 0.6056 Ha
Gap HOMO-LUMO: 0.9968 Ha = 27.12 eV


A água tem 10 elétrons em 5 orbitais duplamente ocupados; o primeiro orbital (~−20,24 Ha) é o caroço 1s do oxigênio. O *gap* HOMO–LUMO será uma grandeza central no Projeto Âncora.

> 💡 O método `mf.analyze()` imprime um relatório completo (energias, ocupações, cargas, dipolo) num único comando.

## 2.6 Mãos à Obra: do átomo de hidrogênio à molécula H₂

**Objetivo:** calcular a energia do átomo de H e da molécula H₂, e observar a energia do H se aproximar do valor exato (−0,5 Ha) conforme a base melhora.

### Parte A — O átomo de hidrogênio

Um único elétron ⇒ camada aberta: `spin=1` e `ROHF`.

In [9]:
from pyscf import gto, scf

for base in ['sto-3g', 'cc-pvdz', 'cc-pvtz']:
    mol = gto.M(atom='H 0 0 0', basis=base, spin=1, verbose=0)
    energia = scf.ROHF(mol).kernel()
    print("%-8s -> %.6f Ha" % (base, energia))

sto-3g   -> -0.466582 Ha
cc-pvdz  -> -0.499278 Ha
cc-pvtz  -> -0.499810 Ha


A energia desce rumo a −0,5 Ha (valor exato). Com STO-3G o erro é ~7%; com cc-pVTZ, ~0,04%. A química quântica computacional é, em boa medida, a arte de controlar esse erro.

### Parte B — A molécula H₂

Camada fechada: `spin=0` e `RHF`.

In [10]:
from pyscf import gto, scf

mol = gto.M(atom='H 0 0 0; H 0 0 0.74', basis='cc-pvdz', verbose=0)
mf = scf.RHF(mol)
energia = mf.kernel()

print("Energia da H2: %.6f Ha" % energia)
print("HOMO: %.4f Ha" % mf.mo_energy[0])

Energia da H2: -1.128700 Ha
HOMO: -0.5924 Ha


Compare com dois átomos de H isolados na mesma base ($2 \times -0{,}499 = -0{,}998$ Ha): a H₂ é bem **mais** estável — a diferença é, essencialmente, a energia da ligação química.

> ⚠️ **Camada fechada vs. aberta.** Rodar `scf.RHF` num átomo de H com `spin=1` gera erro (RHF pressupõe elétrons emparelhados). Regra prática: **conte os elétrons desemparelhados e informe-os em `spin`**.

## 🔷 Projeto Âncora — Etapa 0 (conclusão): o primeiro cálculo do átomo de silício

O átomo de silício é o alicerce do nosso nanocristal. Silício tem 14 elétrons e configuração $[\mathrm{Ne}]\,3s^2\,3p^2$; os dois elétrons $3p$ ocupam orbitais distintos (estado fundamental $^3P$) ⇒ **dois elétrons desemparelhados**: `spin=2`.

In [11]:
from pyscf import gto, scf

# Atomo de silicio isolado: 14 eletrons, estado fundamental 3P (spin=2)
mol_si = gto.M(atom='Si 0 0 0', basis='sto-3g', spin=2, verbose=0)
print("Eletrons:", mol_si.nelectron, " AOs:", mol_si.nao)

mf_si = scf.ROHF(mol_si)
e_si = mf_si.kernel()
print("Energia do atomo de Si (STO-3G): %.6f Ha" % e_si)

Eletrons: 14  AOs: 9
Energia do atomo de Si (STO-3G): -285.466211 Ha


Repetindo com a base `def2-svp` (mais precisa), a energia desce — coerente com o princípio variacional. Anote os dois valores.

In [12]:
mol_si2 = gto.M(atom='Si 0 0 0', basis='def2-svp', spin=2, verbose=0)
e_si2 = scf.ROHF(mol_si2).kernel()
print("Si def2-svp: %.6f Ha  (%d AOs)" % (e_si2, mol_si2.nao))
print("Si sto-3g  : %.6f Ha  (%d AOs)" % (e_si, mol_si.nao))
print("Diferenca  : %.6f Ha  -> base maior, energia mais baixa (variacional)"
      % (e_si2 - e_si))

Si def2-svp: -288.749819 Ha  (18 AOs)


Si sto-3g  : -285.466211 Ha  (9 AOs)
Diferenca  : -3.283608 Ha  -> base maior, energia mais baixa (variacional)


**Marco atingido.** A Parte 0 está completa: ambiente funcional, fluxo de trabalho do PySCF dominado e primeiro cálculo do átomo protagonista do livro. Na Parte I começaremos a **construir o nanocristal**.

## Exercícios

Resolva nas células abaixo.

**2.1 Camada fechada.** Energia HF do átomo de hélio (`He`, `spin=0`, `RHF`) em STO-3G e cc-pVTZ. Compare com −2,86 Ha (limite HF).

**2.2 O papel do `spin`.** Tente `scf.RHF` no átomo de H com `spin=0`. Leia o erro e explique. Depois corrija com `spin=1` e `ROHF`.

**2.3 Curva de dissociação.** Para distâncias H–H de 0,5 a 2,0 Å (passo 0,25), calcule e imprima a energia RHF/STO-3G da H₂. Em qual distância a energia é mínima?

**2.4 Inspecionando orbitais.** Para a água (Seção 2.5), confirme que há 5 orbitais ocupados e calcule o *gap* HOMO–LUMO em eV.

**2.5 Efeito da base no gap.** *Gap* HOMO–LUMO da H₂ em STO-3G e cc-pVDZ. Aumenta ou diminui? Por quê?

**2.6 Projeto Âncora.** Calcule o átomo de Si (`spin=2`, `ROHF`) em `sto-3g` e `def2-svp`. Registre energia, `nao` e por que a base maior dá energia mais baixa.

In [13]:
# 2.1 Camada fechada (He)

In [14]:
# 2.3 Curva de dissociacao da H2

In [15]:
# 2.4 Inspecionando orbitais da agua

In [16]:
# 2.5 Efeito da base no gap